# Feature Engineering


In [1]:
import os
from pathlib import Path
import sys


PROJECT_ROOT = Path(os.getcwd()).parent 

sys.path.append(str(PROJECT_ROOT))

In [2]:
%store -r df_panel

### 1. Biến phụ thuộc Y (Dependent Variables)
- `NPL_ratio` (Tỷ lệ nợ xấu - %): Được tính bằng công thức:
  $$NPL\_ratio_{i,t} = \frac{NPL\_p9\_annual_{i,t} + NPL\_na\_annual_{i,t}}{ASSET\_avg_{i,t}} \times 100$$
  Trong đó:
  - `NPL_p9_annual`: Khoản cho vay quá hạn 90 ngày trở lên nhưng vẫn tính lãi.
  - `NPL_na_annual`: Tài sản nợ không sinh lời (nonaccrual).
  - `ASSET_avg`: Tổng tài sản bình quân năm của ngân hàng.
- `Z_score` (Chỉ số an toàn): Đo lường khoảng cách đến khả năng vỡ nợ, được tính bằng công thức:
  $$Z\_score_{i,t} = \frac{ROA\_annual_{i,t} + CAR\_ratio_{i,t} \times 100}{ROA\_sd3_{i,t}}$$
  Trong đó:
  - `ROA_annual`: Tỷ suất sinh lời trung bình năm trên tổng tài sản (%).
  - `CAR_ratio`: Tỷ lệ an toàn vốn trên tổng tài sản dạng thập phân (Vốn chủ sở hữu / Tổng tài sản).
  - `ROA_sd3`: Độ lệch chuẩn của ROA tính trên cửa sổ trượt 3 năm gần nhất.
- `ln_Zscore`: Logarit tự nhiên của Z-score để phân phối biến tiệm cận phân phối chuẩn.

### 2. Biến độc lập X (Independent Climate Variables)
- `HEAT_DAYS` (Số ngày sốc nhiệt - tương ứng với `HEAT_SHOCK_DAYS` trong lý thuyết): Số ngày trong năm $t$ có nhiệt độ tối đa vượt ngưỡng phân vị thứ 90 (P90) lịch sử của mã ZIP đặt trụ sở ngân hàng.
- `TMAX_AVG` (Nhiệt độ tối đa trung bình - tương ứng với `HEAT_SHOCK_INTENSITY` trong lý thuyết): Nhiệt độ tối đa trung bình hàng ngày trong năm $t$ (°C).
- `HEAT_SHOCK_DUMMY` (Biến giả sốc nhiệt): Được xác định bằng 1 nếu số ngày sốc nhiệt `HEAT_DAYS` > 0, ngược lại bằng 0 (biến này được tạo động trong quá trình hồi quy/phân tích đặc trưng).
- **Biến trễ (Lagged Climate Variables)**: Được tạo ra trong quá trình hồi quy để đánh giá độ trễ tác động (ví dụ: `HEAT_DAYS_(t-1)` hoặc `HEAT_DAYS_lag1`, `TMAX_AVG_lag1`).

### 3. Biến kiểm soát (Control Variables)

#### ** BANK LEVEL CONTROL (Kiểm soát cấp ngân hàng) **
- `SIZE` (Quy mô ngân hàng): Logarit tự nhiên của tổng tài sản trung bình quy về đơn vị USD thực tế:
  $$SIZE_{i,t} = \ln(ASSET\_avg_{i,t} \times 1000)$$
  *Lưu ý:* Biến này được Winsorize ở mức 1% - 99% để hạn chế tác động của giá trị ngoại lai cực đoan.
- `CAR` (Capital Adequacy Ratio - %): Tỷ lệ an toàn vốn (Leverage Ratio) thực tế:
  $$CAR_{i,t} = \frac{CAR\_annual_{i,t}}{ASSET\_avg_{i,t}} \times 100$$
- `NIM` (Net Interest Margin - %): Biên tỷ suất lợi nhuận thuần từ lãi ròng:
  $$NIM_{i,t} = \frac{NIM\_annual_{i,t}}{ASSET\_avg_{i,t}} \times 100$$
- `ROA_annual` (Return on Assets - %): Tỷ suất sinh lời trên tổng tài sản.
- `LTD` (Loan-to-Deposit Ratio): Tỷ lệ dư nợ trên huy động vốn (một số năm có thể bị khuyết dữ liệu).
- `LIQ` (Liquidity Ratio): Tỷ lệ thanh khoản của ngân hàng.
LTD và LIQ tạm thời chưa cho vào do chưa có data

#### ** MACRO LEVEL CONTROL (Kiểm soát vĩ mô) **
*Lưu ý:* Các biến kiểm soát vĩ mô không xuất hiện trực tiếp trong bảng dữ liệu thực tế (`final_research_panel.csv`) mà được kiểm soát gián tiếp thông qua việc sử dụng **Time Fixed Effects ($\gamma_t$)** trong mô hình hồi quy bảng (Panel Fixed Effects), hoặc có thể được bổ sung từ nguồn dữ liệu ngoài của ngân hàng trung ương/bang khi phân tích hồi quy nâng cao. Các biến lý thuyết bao gồm:
- `GDP_GROWTH`: Tăng trưởng GDP bang/địa phương đặt trụ sở ngân hàng.
- `UNEMPLOYMENT`: Tỷ lệ thất nghiệp địa phương.
- `INFLATION`: Tỷ lệ lạm phát quốc gia.



In [3]:
from src.feature_engineering.target_variables import calculate_financial_features
df_calculated = calculate_financial_features(df_panel)

⏳ Đang tính toán các biến tài chính...
✔️ Đã tính toán xong các biến tài chính (SIZE, CAR, NPL_ratio, Z_score).


In [4]:
df_calculated

,CERT,year,ZIP,STALP,NAME,CITY,HEAT_DAYS,TMAX_AVG,HEAT_SHOCK_DUMMY,ASSET,...,longitude,SIZE,CAR_ratio,CAR,ROA_annual,NIM_annual,NPL_ratio,ROA_sd3,Z_score,ln_Zscore
2,2413,2010,78130,TX,First State Bank,New Braunfels,12,25.279178,1,265803.00,...,-98.0742,19.398266,0.097370,9.737004,1.478374,7242.000000,0.549373,0.136437,82.201460,4.409173
3,2413,2011,78130,TX,First State Bank,New Braunfels,82,27.663014,1,269938.25,...,-98.0742,19.413704,0.104107,10.410714,1.593003,7649.250000,1.078487,0.198613,60.437457,4.101609
4,2413,2012,78130,TX,First State Bank,New Braunfels,30,27.043716,1,262885.00,...,-98.0742,19.387227,0.103702,10.370187,1.308460,5956.666667,0.456600,0.143164,81.574857,4.401521
7,2429,2010,23487,VA,Farmers Bank,Windsor,64,20.566849,1,447099.75,...,-76.7324,19.918292,0.080360,8.036014,-1.570221,8395.500000,2.315535,1.160062,5.573657,1.718051
8,2429,2011,23487,VA,Farmers Bank,Windsor,48,21.340548,1,443244.25,...,-76.7324,19.909632,0.084523,8.452338,0.454322,8196.250000,1.787784,1.167621,7.628034,2.031830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3083,90234,2020,1945,MA,Marblehead Bank,Marblehead,48,15.070219,1,241358.50,...,-70.8653,19.301794,0.079655,7.965537,0.223696,4561.000000,0.302973,0.063135,129.708336,4.865288
3084,90234,2021,1945,MA,Marblehead Bank,Marblehead,40,14.713425,1,285435.00,...,-70.8653,19.469525,0.067451,6.745143,0.238244,4523.250000,0.052727,0.055040,126.875108,4.843203
3085,90234,2022,1945,MA,Marblehead Bank,Marblehead,51,15.088493,1,287777.25,...,-70.8653,19.477697,0.068073,6.807348,0.112663,4918.500000,0.036313,0.068691,100.740181,4.612545
3086,90234,2023,1945,MA,Marblehead Bank,Marblehead,37,14.861918,1,262655.50,...,-70.8653,19.386354,0.083577,8.357716,0.592921,6248.500000,0.102987,0.249070,35.936151,3.581744


### Lag Features
* `HEAT_DAYS_LAG1`: Số ngày sốc nhiệt trong năm t-1
* `HEAT_DAYS_LAG2`: Số ngày sốc nhiệt trong năm t-2
* `TMAX_AVG_LAG1`: Nhiệt độ tối đa trung bình hàng ngày trong năm t-1

In [5]:
from src.feature_engineering.lag_features import generate_climate_lag_features

df_final = generate_climate_lag_features(df_calculated)

✔️ Đã tạo thành công các biến trễ (Lag 1 & Lag 2) cho dữ liệu khí hậu.


In [11]:
df_final


,CERT,year,ZIP,STALP,NAME,CITY,HEAT_DAYS,TMAX_AVG,HEAT_SHOCK_DUMMY,ASSET,...,CAR,ROA_annual,NIM_annual,NPL_ratio,ROA_sd3,Z_score,ln_Zscore,HEAT_DAYS_LAG1,HEAT_DAYS_LAG2,TMAX_AVG_LAG1
0,2413,2010,78130,TX,First State Bank,New Braunfels,12,25.279178,1,265803.00,...,9.737004,1.478374,7242.000000,0.549373,0.136437,82.201460,4.409173,NaN,NaN,NaN
1,2413,2011,78130,TX,First State Bank,New Braunfels,82,27.663014,1,269938.25,...,10.410714,1.593003,7649.250000,1.078487,0.198613,60.437457,4.101609,12.0,NaN,25.279178
2,2413,2012,78130,TX,First State Bank,New Braunfels,30,27.043716,1,262885.00,...,10.370187,1.308460,5956.666667,0.456600,0.143164,81.574857,4.401521,82.0,12.0,27.663014
3,2429,2010,23487,VA,Farmers Bank,Windsor,64,20.566849,1,447099.75,...,8.036014,-1.570221,8395.500000,2.315535,1.160062,5.573657,1.718051,NaN,NaN,NaN
4,2429,2011,23487,VA,Farmers Bank,Windsor,48,21.340548,1,443244.25,...,8.452338,0.454322,8196.250000,1.787784,1.167621,7.628034,2.031830,64.0,NaN,20.566849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2649,90234,2020,1945,MA,Marblehead Bank,Marblehead,48,15.070219,1,241358.50,...,7.965537,0.223696,4561.000000,0.302973,0.063135,129.708336,4.865288,41.0,45.0,13.743836
2650,90234,2021,1945,MA,Marblehead Bank,Marblehead,40,14.713425,1,285435.00,...,6.745143,0.238244,4523.250000,0.052727,0.055040,126.875108,4.843203,48.0,41.0,15.070219
2651,90234,2022,1945,MA,Marblehead Bank,Marblehead,51,15.088493,1,287777.25,...,6.807348,0.112663,4918.500000,0.036313,0.068691,100.740181,4.612545,40.0,48.0,14.713425
2652,90234,2023,1945,MA,Marblehead Bank,Marblehead,37,14.861918,1,262655.50,...,8.357716,0.592921,6248.500000,0.102987,0.249070,35.936151,3.581744,51.0,40.0,15.088493


In [9]:
df_final.shape

(2654, 34)

In [10]:
%store df_final

Stored 'df_final' (DataFrame)
